In [1]:
import torch
import torch.nn as nn

from transformers import BertModel

model_config = {} 
model_config['output_dim'] = 2

bert = BertModel.from_pretrained('bert-base-uncased')
model_config['emb_dim'] = bert.config.to_dict()['hidden_size']
print(model_config['emb_dim'])
#768

class SentenceClassification(nn.Module):
    def __init__(self, **model_config):
        
        super().__init__() 
        
        self.bert = bert
        self.fc = nn.Linear(model_config['emb_dim'], model_config['output_dim'])
        
    def forward(self, x):
        pooled_cls_output = self.bert(x)[1]
        return self.fc(pooled_cls_output)
    
    
# 3. 모델 객체 생성 및 테스트
model = SentenceClassification(**model_config)
print("모델 생성 성공!")

# 가짜 입력 데이터 테스트 (batch_size=2, sequence_length=10)
dummy_input = torch.randint(0, 1000, (2, 10))
output = model(dummy_input)
print("출력 결과 형태(shape):", output.shape) # Expected: torch.Size([2, 2])

/home/centa/miniconda3/envs/firstproject/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 554.20it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   

768
모델 생성 성공!
출력 결과 형태(shape): torch.Size([2, 2])


In [2]:
import sys
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import BertTokenizer, BertModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

1. 데이터 및 BERT 토크나이저 로드 

In [3]:
raw_datasets = load_dataset("imdb")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

2. Dataset 클래스 정의 (커스텀 Dataset 클래스)

In [14]:
class IMDBBertDataset(Dataset):
    # 단어가 늘어날수록 연산량은 제곱으로 폭발해 모델 훈련이 느려져서 줄임 원래 논문 기재 기준 Max_len = 510(512 - 2개 [cls], [sep] 토큰 개수 고려 최신 알고리즘은 필요 x)
    def __init__(self, dataset, tokenizer, max_len=128): 
        self.texts = dataset["text"]
        self.labels = dataset["label"]
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            # output_dim=2일 때는 CrossEntropyLoss를 쓰므로 dtype=torch.long 
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

3. Split 및 DataLoader 생성

In [5]:
split = raw_datasets["train"].train_test_split(test_size=0.2, seed=42)

train_ds = IMDBBertDataset(split["train"], tokenizer)
valid_ds = IMDBBertDataset(split["test"], tokenizer)
test_ds = IMDBBertDataset(raw_datasets["test"], tokenizer)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=8)
test_loader = DataLoader(test_ds, batch_size=8)

4. BERT 분류 모델 정의

In [6]:
model_config = {} 
model_config['output_dim'] = 2  # 긍정/부정 2개 클래스 분류

bert = BertModel.from_pretrained('bert-base-uncased')
model_config['emb_dim'] = bert.config.to_dict()['hidden_size']

class SentenceClassification(nn.Module):
    def __init__(self, **model_config):
        super().__init__() 
        self.bert = bert
        self.fc = nn.Linear(model_config['emb_dim'], model_config['output_dim'])
        
    # attention_mask를 받아 안전하게 BERT에 전달하도록 확장
    def forward(self, x, attention_mask=None):
        if attention_mask is not None:
            outputs = self.bert(input_ids=x, attention_mask=attention_mask)
        else:
            outputs = self.bert(x)
            
        pooled_cls_output = outputs[1] # 아까 쓰신 [1]번 인덱스 사용 방식 그대로
        return self.fc(pooled_cls_output)

model = SentenceClassification(**model_config).to(device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 488.79it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


5. 학습(Train) 및 검증(Eval) 함수 정의

In [7]:
def train(model, iterator, optimizer, loss_fn, device):
    epoch_loss = 0
    epoch_acc = 0
    model.train()

    for idx, batch in enumerate(iterator):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device).long()

        optimizer.zero_grad()
        predictions = model(input_ids, attention_mask)
        
        loss = loss_fn(predictions, labels)
        preds = torch.argmax(predictions, dim=1)
        acc = (preds == labels).float().mean()

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += acc.item()

        sys.stdout.write(f"\r[Train] Batch {idx+1}/{len(iterator)} | Loss: {loss.item():.4f} | Acc: {acc.item():.4f}")

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

In [8]:
def evaluate(model, iterator, loss_fn, device):
    epoch_loss = 0
    epoch_acc = 0
    model.eval()

    with torch.no_grad():
        for batch in iterator:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device).long()

            predictions = model(input_ids, attention_mask)
            loss = loss_fn(predictions, labels)

            preds = torch.argmax(predictions, dim=1)
            acc = (preds == labels).float().mean()

            epoch_loss += loss.item()
            epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

6. 손실함수/옵티마이저 설정 & Epoch 돌리기

In [9]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)
loss_fn = nn.CrossEntropyLoss().to(device)

N_EPOCHS = 3
best_valid_loss = float('inf')

for epoch in range(N_EPOCHS):
    train_loss, train_acc = train(model, train_loader, optimizer, loss_fn, device)
    print(f"\nEpoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%")

    valid_loss, valid_acc = evaluate(model, valid_loader, loss_fn, device)
    print(f"Epoch {epoch+1:02d} | Valid Loss: {valid_loss:.4f} | Valid Acc: {valid_acc*100:.2f}%")

    # 검증 손실이 가장 낮을 때 모델 저장
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), './BERT_model.pt')
        print("--> 최고 성능 모델 저장 완료!")

[Train] Batch 2500/2500 | Loss: 0.2113 | Acc: 0.8750
Epoch 01 | Train Loss: 0.3542 | Train Acc: 84.06%
Epoch 01 | Valid Loss: 0.3188 | Valid Acc: 86.16%
--> 최고 성능 모델 저장 완료!
[Train] Batch 2500/2500 | Loss: 0.2188 | Acc: 0.8750
Epoch 02 | Train Loss: 0.2039 | Train Acc: 92.03%
Epoch 02 | Valid Loss: 0.2987 | Valid Acc: 88.12%
--> 최고 성능 모델 저장 완료!
[Train] Batch 2500/2500 | Loss: 0.0064 | Acc: 1.0000
Epoch 03 | Train Loss: 0.1065 | Train Acc: 96.12%
Epoch 03 | Valid Loss: 0.4153 | Valid Acc: 87.26%


7.  테스트 데이터로 최종 평가

In [10]:
model.load_state_dict(torch.load('./BERT_model.pt'))
test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)
print(f"\n[최종 Test 결과] Loss: {test_loss:.4f} | Acc: {test_acc*100:.2f}%")


[최종 Test 결과] Loss: 0.3005 | Acc: 87.72%


8. 저장한 모델 불러와 실제 입력 테스트 

In [12]:
import torch
from transformers import BertTokenizer

# 1. 실행 장치 설정 (GPU / MPS / CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# 2. 작성하셨던 모델 클래스 구조 그대로 생성 및 가중치 불러오기
model = SentenceClassification(**model_config)
model.load_state_dict(torch.load("./BERT_model.pt", map_location=device))
model.to(device)
model.eval()  # ⚠️ 테스트 시 드롭아웃 등 비활성화 (필수)

# 3. 토크나이저 준비
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

print("✅ ./BERT_model.pt 불러오기 성공!")

# 4. 리뷰 테스트 함수
def test_review(text):
    # 토크나이저 전처리
    encoding = tokenizer(
        text,
        max_length=128,  # 학습 때 설정했던 max_len
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    
    # 모델 예측
    with torch.no_grad():
        logits = model(input_ids, attention_mask=attention_mask)
        probs = torch.softmax(logits, dim=1)
        pred_class = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred_class].item() * 100
        
    result = "긍정 😃" if pred_class == 1 else "부정 😡"
    print(f"리뷰: \"{text}\"")
    print(f"결과: {result} (확확률: {confidence:.2f}%)\n")

# --- 🧪 원하는 문장으로 테스트 ---
test_review("This movie was absolute perfection. I loved every minute of it!")
test_review("Terrible acting and horrible story. Total waste of my time.")

✅ ./BERT_model.pt 불러오기 성공!
리뷰: "This movie was absolute perfection. I loved every minute of it!"
결과: 긍정 😃 (확확률: 99.35%)

리뷰: "Terrible acting and horrible story. Total waste of my time."
결과: 부정 😡 (확확률: 99.76%)

